In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
df = pd.read_csv("/Users/sherry/Desktop/HACKATHON/zomato_kpt__Final_Dataset2.csv")

In [36]:
df.head()

,order_time,restaurant_id,cuisine_type,restaurant_capacity,restaurant_rating,chef_count,efficiency_score,total_items,item_complexity_score,active_orders_in_kitchen,queue_length,kitchen_utilization,weather,festival_flag,hour,weekend,prep_time
0,2024-01-01 00:00:00,37,Biryani,10,3.6,5,0.876097,4,6.24,0,0.0,0.0,Rainy,1,0,0,25.77
1,2024-01-01 00:00:00,198,Continental,13,3.0,8,0.812020,6,14.66,0,0.0,0.0,Rainy,1,0,0,55.89
2,2024-01-01 00:00:00,50,Fast Food,5,4.5,3,0.994893,4,4.46,0,0.0,0.0,Rainy,1,0,0,8.59
3,2024-01-01 00:00:00,187,Biryani,7,4.1,4,0.978308,5,11.80,0,0.0,0.0,Rainy,1,0,0,33.36
4,2024-01-01 00:00:00,129,North Indian,5,4.1,3,0.980710,2,3.63,0,0.0,0.0,Rainy,1,0,0,24.25


In [ ]:


# =====================================================
# 0️⃣ BASIC PREPARATION
# =====================================================

# Convert to datetime
df["order_time"] = pd.to_datetime(df["order_time"])

# Sort properly for time-based operations
df = df.sort_values(["restaurant_id", "order_time"]).reset_index(drop=True)

# =====================================================
# 1️⃣ REMOVE WEAK / REDUNDANT FEATURES
# =====================================================

# Remove rating (weak causal link)
if "restaurant_rating" in df.columns:
    df.drop(columns=["restaurant_rating"], inplace=True)

# Remove derived utilization if redundant
if "kitchen_utilization" in df.columns:
    df.drop(columns=["kitchen_utilization"], inplace=True)

# =====================================================
# 2️⃣ CONGESTION FEATURES (CRITICAL)
# =====================================================

df["chef_load_ratio"] = (
    df["active_orders_in_kitchen"] / df["chef_count"]
)

df["queue_per_chef"] = (
    df["queue_length"] / df["chef_count"]
)

df["capacity_utilization"] = (
    df["active_orders_in_kitchen"] / df["restaurant_capacity"]
)

# =====================================================
# 3️⃣ SERVICE CAPACITY FEATURES
# =====================================================

df["effective_capacity"] = (
    df["chef_count"] * df["efficiency_score"]
)

df["effective_load_ratio"] = (
    df["active_orders_in_kitchen"] / df["effective_capacity"]
)

# =====================================================
# 4️⃣ LEAKAGE-SAFE RESTAURANT BASELINE SPEED
# =====================================================

df["restaurant_avg_prep_time"] = (
    df.groupby("restaurant_id")["prep_time"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

df["restaurant_std_prep_time"] = (
    df.groupby("restaurant_id")["prep_time"]
    .expanding()
    .std()
    .shift(1)
    .reset_index(level=0, drop=True)
)

# Fill first few NaNs with global mean
global_mean = df["prep_time"].mean()
df["restaurant_avg_prep_time"].fillna(global_mean, inplace=True)
df["restaurant_std_prep_time"].fillna(0, inplace=True)

# =====================================================
# 5️⃣ RECENT PERFORMANCE (LEAKAGE SAFE)
# =====================================================

df["prep_time_recent_avg"] = (
    df.groupby("restaurant_id")["prep_time"]
    .shift(1)
    .rolling(10, min_periods=1)
    .mean()
)

df["prep_time_recent_std"] = (
    df.groupby("restaurant_id")["prep_time"]
    .shift(1)
    .rolling(10, min_periods=1)
    .std()
)

df["prep_time_recent_std"].fillna(0, inplace=True)

# =====================================================
# 6️⃣ COMPLEXITY INTERACTIONS
# =====================================================

df["complexity_load_interaction"] = (
    df["item_complexity_score"] *
    df["chef_load_ratio"]
)

df["complexity_queue_interaction"] = (
    df["item_complexity_score"] *
    df["queue_per_chef"]
)

# =====================================================
# 7️⃣ TIME FEATURES (CORRECTED)
# =====================================================

# Extract time features properly
df["hour"] = df["order_time"].dt.hour
df["day_of_week"] = df["order_time"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

# Hourly demand intensity (group BEFORE dropping hour)
df["hourly_avg_load"] = (
    df.groupby("hour")["active_orders_in_kitchen"]
    .transform("mean")
)

# Cyclical encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# Drop raw time columns
df.drop(columns=["hour", "day_of_week"], inplace=True)

# =====================================================
# 8️⃣ LOG TRANSFORM SKEWED FEATURES
# =====================================================

df["log_queue_length"] = np.log1p(df["queue_length"])
df["log_active_orders"] = np.log1p(df["active_orders_in_kitchen"])

# =====================================================
# 9️⃣ SAFE TARGET ENCODING FOR CUISINE
# =====================================================

df["cuisine_encoded"] = (
    df.groupby("cuisine_type")["prep_time"]
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

df["cuisine_encoded"].fillna(global_mean, inplace=True)

# =====================================================
# 🔟 FINAL CLEANUP (OPTIONAL)
# =====================================================

# You may optionally drop redundant raw columns:
# df.drop(columns=["queue_length"], inplace=True)
# df.drop(columns=["active_orders_in_kitchen"], inplace=True)

print("Final dataset shape:", df.shape)
print(df.head())

In [42]:
df.to_csv("zomato_kpt_dataset_final_training.csv", index=False)

In [39]:
print(df.columns)

Index(['order_time', 'restaurant_id', 'cuisine_type', 'restaurant_capacity',
       'chef_count', 'efficiency_score', 'total_items',
       'item_complexity_score', 'active_orders_in_kitchen', 'queue_length',
       'weather', 'festival_flag', 'weekend', 'prep_time', 'chef_load_ratio',
       'queue_per_chef', 'capacity_utilization', 'effective_capacity',
       'effective_load_ratio', 'restaurant_avg_prep_time',
       'restaurant_std_prep_time', 'prep_time_recent_avg',
       'prep_time_recent_std', 'complexity_load_interaction',
       'complexity_queue_interaction', 'is_weekend', 'hourly_avg_load',
       'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'log_queue_length',
       'log_active_orders', 'cuisine_encoded'],
      dtype='object')


In [41]:
numeric_df = df.select_dtypes(include=np.number)

corr = numeric_df.corr()["prep_time"].sort_values(ascending=False)

print(corr)

prep_time                       1.000000
prep_time_recent_avg            0.762128
restaurant_avg_prep_time        0.723460
complexity_load_interaction     0.634895
cuisine_encoded                 0.600794
capacity_utilization            0.593978
effective_load_ratio            0.593214
chef_load_ratio                 0.586047
queue_length                    0.550658
queue_per_chef                  0.549941
log_queue_length                0.529319
complexity_queue_interaction    0.528373
active_orders_in_kitchen        0.523272
restaurant_std_prep_time        0.508110
item_complexity_score           0.449038
log_active_orders               0.424730
total_items                     0.391786
prep_time_recent_std            0.193120
hourly_avg_load                 0.145866
is_weekend                      0.047647
weekend                         0.047647
hour_cos                        0.045623
dow_cos                         0.009713
festival_flag                   0.002655
efficiency_score